In [0]:
%run "../utils/00_config"

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================
# EXECUTION PARAMETERS
# ==========================================

EXECUTION_DATE = "2026-11-21"
EXECUTION_ID = "ae7db7b4-20af-488a-9253-db808ee32106"

# ==========================================
# SOURCE PATHS
# ==========================================

EXECUTION_RAW_PATH = (
    f"{RAW_PATH}"
    f"execution_date={EXECUTION_DATE}/"
    f"execution_id={EXECUTION_ID}/"
)

MANIFEST_PATH = EXECUTION_RAW_PATH + "manifest.json"
RESPONSES_PATH = EXECUTION_RAW_PATH + "raw_api_responses.jsonl.gz"
ITEM_EXECUTION_PATH = EXECUTION_RAW_PATH + "item_execution.jsonl.gz"

# ==========================================
# TARGET TABLES
# ==========================================

TABLE_RAW_RESPONSES = get_table(
    "bronze", "raw_api_responses"
)

TABLE_ITEM_EXECUTION = get_table(
    "bronze", "item_execution"
)

print(f"Environment: {ENV.upper()}")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Source path: {EXECUTION_RAW_PATH}")

In [0]:
# ==========================================
# FILE VERIFICATION
# ==========================================

files = {
    file.name.rstrip("/")
    for file in dbutils.fs.ls(EXECUTION_RAW_PATH)
}

required_files = {
    "manifest.json",
    "raw_api_responses.jsonl.gz",
    "item_execution.jsonl.gz"
}

missing_files = required_files - files

if missing_files:
    raise FileNotFoundError(
        f"Missing required files: {missing_files}"
    )

# ==========================================
# DATA LOADING
# ==========================================

manifest_df = (
    spark.read
    .option("multiLine", True)
    .json(MANIFEST_PATH)
)

raw_responses = spark.read.json(RESPONSES_PATH)

item_execution = spark.read.json(ITEM_EXECUTION_PATH)

manifest = manifest_df.first().asDict(recursive=True)

print("Source files loaded successfully.")

In [0]:
print(f"Manifest status: {manifest['status']}")
print(f"Raw responses: {raw_responses.count()}")
print(f"Item executions: {item_execution.count()}")

In [0]:
# ==========================================
# EXECUTION ID VALIDATION
# ==========================================

for name, df in {
    "raw_api_responses": raw_responses,
    "item_execution": item_execution
}.items():

    invalid_execution = df.filter(
        F.col("execution_id").isNull()
        | (F.col("execution_id") != EXECUTION_ID)
    ).count()

    if invalid_execution:
        raise ValueError(
            f"{name}: invalid execution_id found."
        )

# ==========================================
# PRIMARY KEY VALIDATION
# ==========================================

def validate_unique_key(df, columns, table_name):

    duplicates = (
        df.groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates:
        raise ValueError(
            f"Duplicate key found in {table_name}: {columns}"
        )


validate_unique_key(
    raw_responses,
    ["execution_id", "event_type", "resource_slug"],
    "raw_api_responses"
)

validate_unique_key(
    item_execution,
    ["execution_id", "item_slug"],
    "item_execution"
)

print("DataFrame validations passed.")

In [0]:
# ==========================================
# PERSISTENCE FUNCTION
# ==========================================

def merge_bronze(df, table_name, keys):

    if not spark.catalog.tableExists(table_name):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        print(f"Table created: {table_name}")

    else:

        condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in keys]
        )

        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(
                df.alias("source"),
                condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Table updated: {table_name}")


# ==========================================
# BRONZE PERSISTENCE
# ==========================================

merge_bronze(
    raw_responses,
    TABLE_RAW_RESPONSES,
    ["execution_id", "event_type", "resource_slug"]
)

merge_bronze(
    item_execution,
    TABLE_ITEM_EXECUTION,
    ["execution_id", "item_slug"]
)

In [0]:
# ==========================================
# READ PERSISTED TABLES
# ==========================================

bronze_responses = spark.table(TABLE_RAW_RESPONSES)

bronze_execution = spark.table(TABLE_ITEM_EXECUTION)

# ==========================================
# EXECUTION RECORD COUNTS
# ==========================================

final_responses = bronze_responses.filter(
    F.col("execution_id") == EXECUTION_ID
).count()

final_items = bronze_execution.filter(
    F.col("execution_id") == EXECUTION_ID
).count()

# ==========================================
# FINAL VALIDATION
# ==========================================

print("======================================")
print("BRONZE FINAL VALIDATION")
print("======================================")

print(f"Expected responses: {response_count}")
print(f"Persisted responses: {final_responses}")

print(f"Expected items: {item_count}")
print(f"Persisted items: {final_items}")

if final_responses != response_count:
    raise ValueError(
        "Mismatch in persisted response records."
    )

if final_items != item_count:
    raise ValueError(
        "Mismatch in persisted item records."
    )

print("======================================")
print("BRONZE INGESTION COMPLETED SUCCESSFULLY")
print("======================================")